# Neural PointMaze — Full Training (Colab)

Runs the full 5000-episode 2-phase training with stabilization improvements:
- Phase-aware epsilon reset at Phase 1→2 boundary
- Buffer truncation (remove stale diverse data)
- Cosine LR annealing with warm restart
- Gradient clipping on reward network

**Requires gymnasium-robotics** (installed automatically below).

**Runtime**: ~1-2 hours on CPU, ~30-45 min with GPU

In [1]:
# 1. Install dependencies and clone repo
!pip install -q gymnasium-robotics mujoco torch numpy scipy matplotlib scikit-learn imageio imageio-ffmpeg

# Clone repo
!git clone -b neural-successor-representation https://github.com/PrashRangarajan/Successor_Active_Inference_Clean.git
%cd Successor_Active_Inference_Clean

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.2/26.2 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 120.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.5/852.5 kB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 30.1 MB/s eta 0:00:00
Cloning into 'Successor_Active_Inference_Clean'...
remote: Enumerating objects: 919, done.
remote: Counting objects: 100% (919/919), done.
remote: Compressing objects: 100% (464/464), done.
remote: Total 919 (delta 576), reused 782 (delta 439), pack-reused 0 (from 0)
Receiving objects: 100% (919/919), 24.46 MiB | 9.38 MiB/s, done.
Resolving deltas: 100% (576/576), done.
/content/Successor_Active_Inference_Clean


In [2]:
# 2. Verify imports and PointMaze environment
import sys
sys.path.insert(0, '.')

import torch
import gymnasium as gym
import gymnasium_robotics
import numpy as np

gym.register_envs(gymnasium_robotics)

# Test PointMaze environment
env = gym.make('PointMaze_UMaze-v3')
obs, _ = env.reset()
print(f'Observation keys: {list(obs.keys())}')
print(f'observation shape: {obs["observation"].shape}')
print(f'desired_goal shape: {obs["desired_goal"].shape}')
print(f'Action space: {env.action_space}')
env.close()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nUsing device: {device}')
print(f'PyTorch: {torch.__version__}')

Observation keys: ['observation', 'achieved_goal', 'desired_goal']
observation shape: (4,)
desired_goal shape: (2,)
Action space: Box(-1.0, 1.0, (2,), float32)

Using device: cuda
PyTorch: 2.10.0+cu128


In [ ]:
# 3. Run full training (5000 episodes: 2000 diverse + 3000 mixed)
import torch
device_flag = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device_flag}')
!python -u examples/run_neural_point_maze.py --device {device_flag}

Training on: cuda
Neural Successor Feature Agent -- PointMaze
Observation dim: 6 (6D: x, y, vx, vy, gx, gy)
Actions: 8 (8 directional forces)
SF dim: 64
Maze: PointMaze_UMaze-v3
Bins: 20x20 (112 navigable)
Goal: (-1.25, 1.23)

Device: cuda
Phase 1: Diverse exploration (2000 episodes, 100% diverse)
Learning with Neural SF (2000 episodes, sf_dim=64, ε: 1.000→0.05)...
  Episode 400/2000 | Avg reward: -301.19 | SF loss: 0.0028 | ε: 0.240 | Buffer: 120000
  Episode 800/2000 | Avg reward: -442.02 | SF loss: 0.0047 | ε: 0.050 | Buffer: 240000
  Episode 1200/2000 | Avg reward: -408.22 | SF loss: 0.0250 | ε: 0.050 | Buffer: 300000
  Episode 1600/2000 | Avg reward: -419.38 | SF loss: 0.0532 | ε: 0.050 | Buffer: 300000
  Episode 2000/2000 | Avg reward: -449.22 | SF loss: 0.0866 | ε: 0.050 | Buffer: 300000
Learning complete. Total steps: 600000
  Buffer truncated: 300000 -> 90000 (kept 30%)
  LR reset: SF=1.5e-04, RW=1.5e-04, cosine decay over 900000 steps

Phase 2: Goal-focused (3000 episodes, 30

In [ ]:
# 4. Inspect training diagnostics
import torch
import numpy as np

checkpoint = torch.load('data/neural_point_maze/checkpoint.pt',
                        map_location='cpu', weights_only=False)
print('Checkpoint keys:', list(checkpoint.keys()))
print(f'Total steps: {checkpoint["total_steps"]}')
print(f'Final epsilon: {checkpoint["epsilon"]:.4f}')

In [ ]:
# 5. Plot training curves
import matplotlib.pyplot as plt
import numpy as np

log = checkpoint.get('training_log', {})

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (key, title) in zip(axes.flat, [
    ('sf_loss', 'SF TD Loss'),
    ('reward_loss', 'Reward Prediction Loss'),
    ('episode_reward', 'Episode Reward'),
    ('episode_steps', 'Episode Steps'),
]):
    data = log.get(key, [])
    if data:
        ax.plot(data, alpha=0.3, linewidth=0.5)
        w = min(100, len(data) // 10 + 1)
        if w > 1:
            sm = np.convolve(data, np.ones(w)/w, mode='valid')
            ax.plot(np.arange(w-1, w-1+len(sm)), sm, linewidth=2)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 6. Download checkpoints and results
from google.colab import files
import shutil

shutil.make_archive('neural_pointmaze_results', 'zip', '.', 'data/neural_point_maze')
files.download('neural_pointmaze_results.zip')